### Cluster profiling - using assigned cluster values to create new variables

Cluste profiling seems like target leakage, but we have to think about this carefully:

- We don't know the target value in real-time
- ... but we do know the cluster value based on KMeans
- ... based on the KMeans -based cluster value, we can also know the average target value for each cluster
- ... => no target leakage

However, if anything changes in the actual clustering => averages have to be re-calculated

In [20]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# load data
df = pd.read_csv("winequality-red.csv")
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


In [21]:
# X/y -split
X = df.drop("quality", axis=1)

# most clustering algorithms require scaled values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# create clustering model and train it with our data
kmeans = KMeans(n_clusters=6, random_state=42)

# place the cluster value back to DataFrame
df['cluster'] = kmeans.fit_predict(X_scaled)

df.head(5)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,cluster
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


In [22]:
# let's how the cluster values are spread out
df['cluster'].value_counts()

cluster
0    515
2    339
1    287
5    237
4    193
3     28
Name: count, dtype: int64

In [23]:
# not an optimal clustering, but enough for demonstration purposes
cluster_counts = df.groupby(['quality', 'cluster']).size().unstack(fill_value=0)
print(cluster_counts)

# remember to lock down the random seed when you find a good clustering result

# ideally the clustering should clearly separate different qualities in the target
# currently, it's a bit all over the place

# also remember the original data is mostly concentrated on qualities 5 and 6
# => this is why most values are in qualities 5 and 6

cluster    0    1    2   3    4    5
quality                             
3          7    2    1   0    0    0
4         30    5    5   1   10    2
5        289   88  228  17   37   22
6        171  132   94   9  115  117
7         18   55   11   1   27   87
8          0    5    0   0    4    9


### Perform cluster profiling => calculate averages based on the created clusters

In [24]:
# calculate the average of quality for each cluster
cluster_quality_average = df.groupby('cluster')['quality'].mean()

# place the averages based on clusters back into DataFrame
df['cluster_average_quality'] = df['cluster'].map(cluster_quality_average)

df.head(10)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,cluster,cluster_average_quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,5.316505
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0,5.316505
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0,5.316505
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1,5.864111
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,5.316505
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,0,5.316505
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5,0,5.316505
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7,0,5.316505
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7,0,5.316505
9,7.5,0.50,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5,2,5.321534


In [25]:
# study the average quality for each cluster
# these averages are probably not that helpful for a ML model (mostly same averages)
# this implies => overlap and noise in data
df.groupby("cluster", as_index=False)["cluster_average_quality"].mean()


,cluster,cluster_average_quality
0,0,5.316505
1,1,5.864111
2,2,5.321534
3,3,5.357143
4,4,5.886010
5,5,6.333333
